## **Cropland Area Estimation -- Bungoma County, Kenya (2025 season)**
### **Large-Group Training Variant**

This notebook is a **single-round variant** of `area_estimation_bungoma2025.ipynb`, for training sessions with many participants. It reuses that notebook's map import/reprojection, sample export, and area/accuracy estimation steps unchanged, but replaces the pilot-sample-and-Neyman-allocation sample design with a simpler one sized directly to the training group: every participant annotates a fixed quota of units, in a single annotation round.

Author: Josef Wagner ([University of Strasbourg](https://www.unistra.fr/fr), [NASA Harvest](https://www.nasaharvest.org/)) jwagner@unistra.fr

### **Workflow**

1. Reproject the map to an auto-derived equal-area projection.
2. Set strata and the large-group sample-size parameters (participants x samples-per-participant).
3. Compute pixel counts, then allocate the total sample proportionally across strata (with an optional per-stratum minimum).
4. Draw and export the sample for annotation in STAC Notator (single **checkpoint**).
5. Load the annotated sample back in.
6. Compute the design-based cropland area estimate, with uncertainty.
7. Compute map accuracy (overall, user's, producer's), with uncertainty.

See the **Background** section below (after Step 1) for a step-by-step recap of the statistically rigorous pilot + Neyman process this notebook simplifies, and why.

### **Setup**
First, install packages and import helper functions

In [ ]:
## The tricky part: installing gdal

# Update packages
!apt-get update -qq

# Install GDAL system libraries
!apt-get install -y gdal-bin libgdal-dev

import os
os.environ['CPLUS_INCLUDE_PATH'] = '/usr/include/gdal'
os.environ['C_INCLUDE_PATH'] = '/usr/include/gdal'

%pip install --upgrade pip -q
%pip install numpy pandas shapely fiona geopandas scikit-learn pyproj -q
# Optional: install GDAL Python bindings (match the system version)
%pip install gdal==$(gdal-config --version) -q

from osgeo import gdal, ogr, osr

print("GDAL version:", gdal.__version__)

In [ ]:
def in_colab():
  try:
    import google.colab
    return True
  except ImportError:
    return False

if in_colab():
  # git-lfs is required to fetch the real input_data/*.tif raster (LFS-tracked);
  # without it, the clone would only pull a small LFS pointer stub file.
  !apt-get install -y git-lfs -qq
  !git lfs install
  import os
  repo_dir = 'E2E_StacNotator_AreaEstimation_Training'
  repo_url = 'https://github.com/jowa-ea/E2E_StacNotator_AreaEstimation_Training.git'

  if os.path.basename(os.getcwd()) == repo_dir and os.path.isdir('.git'):
    # Re-running this cell in an already-warm runtime: a prior run's %cd
    # persists across cell re-executions within the same kernel, so we're
    # already sitting inside the clone -- checking for a same-named
    # subfolder (as opposed to checking whether we're already IN it) would
    # find nothing and clone again into ourselves, nesting the repo inside
    # itself. Just pull in place instead.
    !git pull
  elif os.path.isdir(repo_dir):
    # Cloned earlier in this runtime, but this cell hasn't cd'ed into it yet
    # (e.g. re-run after a kernel restart that kept the disk around).
    %cd {repo_dir}
    !git pull
  else:
    !git clone {repo_url}
    %cd {repo_dir}
else:
  print("Running locally - skipping git clone")

In [ ]:
# Import modules and packages
import os
import pandas as pd
import geopandas as gpd
from osgeo import gdal
pd.options.display.float_format = '{:.3f}'.format
gdal.UseExceptions()

import utils_ea_reprojection as ea
import utils_stratified_random_sampling as srs

In [ ]:
## Paths
base_path = os.getcwd() if in_colab() else '.'
input_data_path = os.path.join(base_path, 'input_data')
outputs_path = os.path.join(base_path, 'outputs')
os.makedirs(outputs_path, exist_ok=True)

crop_map_2025 = os.path.join(input_data_path, 'BungomaCropland2025.tif')  # EPSG:4326

#### **Step 1.** Reproject to an equal-area projection

>Pixel counting (hectares per stratum) and equal-probability stratified sampling both assume every pixel covers the same amount of ground area. In geographic coordinates (**EPSG:4326**) that's false: a degree of longitude covers less real ground distance near the poles than at the equator, so a fixed-size pixel in degrees does **not** represent a fixed amount of ground area across (or even within) the raster. Any hectare calculation or "uniformly sample a pixel" procedure done directly on it would be distorted -- unequally sized pixels get an unequal chance of selection, and area sums would be biased.
>
>Reprojecting to an **equal-area projection** removes this distortion: every pixel covers the same ground area everywhere in the raster, so pixel counts translate directly and unbiasedly into hectares, and every pixel gets an equal chance of being sampled.
>
>Rather than hardcoding a CRS for Bungoma specifically, the cell below **auto-derives** a Lambert Azimuthal Equal-Area (LAEA) projection centered on the raster's own bounding-box centroid -- so the same code is reusable for a new season or a new study region without a manual CRS lookup.

In [ ]:
# Auto-derive an equal-area (LAEA) projection centered on the map's own extent, and reproject to it.
# Nearest-neighbour resampling is required here: the raster is categorical (0/1 class codes),
# and any other resampling method would blend/interpolate those codes into meaningless values.
proj_str = ea.derive_ea_proj_string(
    crop_map_2025, out_proj_path=os.path.join(outputs_path, 'bungoma2025_ea_proj.txt')
)
print('Auto-derived equal-area projection:\n', proj_str)

crop_map_2025_ea = os.path.join(outputs_path, 'BungomaCropland2025_ea.tif')
ea.raster_to_ea(crop_map_2025, crop_map_2025_ea, proj_str, resampling_method='nearest')

#### **Background** -- the statistically rigorous process, and why this notebook simplifies it

>`area_estimation_bungoma2025.ipynb` (and its script equivalent, `main.py`) implements the statistically rigorous version of this design, in two annotation rounds:
>
>1. **Pilot sample.** Draw a small, proportionally-allocated stratified random sample (`pilot_n` units) directly from the map -- proportional allocation is a neutral default here, since no per-stratum variance estimate exists yet.
>2. **Pilot annotation.** Interpret the pilot sample in STAC Notator to get its first reference labels (*round-1 checkpoint*).
>3. **Neyman priors (Sh).** From the annotated pilot's per-stratum user's accuracy, estimate a prior standard deviation `Sh = sqrt(Ui*(1-Ui))` for each stratum.
>4. **Neyman sample size (n_tot).** Combine `Sh` with a user-set target coefficient of variation (`cv_target`) and confidence level to compute the *total* sample size needed to hit that accuracy threshold: `n_tot = z^2 * (sum_h Wh*Sh)^2 / E^2`.
>5. **Neyman allocation (n_h).** Split `n_tot` across strata in proportion to each stratum's contribution to overall variance (`Wh*Sh`), so large, uncertain strata get proportionally more units -- this is what makes the design *optimal* for a given `n_tot`.
>6. **Full sample, nested with the pilot.** Draw the full `n_h`-per-stratum sample with the *same random seed* as the pilot, so the pilot's own sampled units -- and once annotated, their labels -- are reused rather than re-drawn or re-annotated.
>7. **Round-2 annotation.** Only the units NOT already covered by the pilot are exported for a second annotation round (*round-2 checkpoint*).
>8. **Combine both rounds** into one fully annotated sample.
>9. **Design-based area and accuracy estimates**, with uncertainty (Olofsson et al., 2014).
>
>That design is optimal because `n_tot` and its allocation are *derived from data* (the pilot's own observed variances) to hit a chosen accuracy target at the smallest possible sample size. The cost is two separate annotation rounds, with a pause between them while `Sh`/`n_tot`/allocation get computed -- workable for a single analyst or small team, but a poor fit for a **large training group**: pausing dozens of participants mid-session to recompute an allocation, then sending out a second, differently-sized batch, adds coordination overhead disproportionate to the training's purpose.
>
>**This notebook trades the statistically-optimal Neyman design for a single-round design sized to the group itself:**
>- No pilot round, no annotated-pilot variances, no Neyman sample-size formula or Neyman allocation.
>- The total sample size (`Ni`) is set directly from classroom capacity: `Ni = n_participants * n_samples_per_participant` -- every participant annotates the same fixed quota, once.
>- `Ni` is allocated across strata **proportionally** to stratum area (`Wh`) -- the same neutral allocation the pilot itself uses above -- optionally with a **minimum per stratum** (`min_allocation`) so a small stratum still gets enough units for a meaningful accuracy check, rather than being rounded down to (near) zero.
>- One annotation round, one export, one import -- no nesting, no reconciliation, no combining two files.
>
>The resulting sample size is *not* tuned to hit a target CV the way the pilot + Neyman design's is -- it is whatever `n_participants * n_samples_per_participant` happens to be. Report the resulting precision (Step 6 below) as an outcome of the design actually used, not a pre-committed target.

#### **Step 2.** Strata, target stratum, and the large-group sample-size parameters

>Strata are the same as in the standard workflow: **non-cropland** (0) and **cropland** (1).
>
>`n_participants` and `n_samples_per_participant` are what replace `cv_target`/`pilot_n` here -- set them to the actual training-group size and how many units each participant should annotate; `Ni` (the total sample size) is computed from them in Step 4.
>
>`min_allocation` is the optional floor (in sample units) applied to *every* stratum during proportional allocation -- raise it above 0 if a small stratum would otherwise get very few (or zero) units.
>
>`confidence` is still used to report the final area estimate's precision (Step 7), even though it no longer drives the sample size.
>
>`seed` is reused for the sample draw and the row shuffle before export, for reproducibility.
>
>`id_col`/`true_col` and `STRATUM_TRUE_LABELS` mean the same as in the standard notebook's Step 2 -- they describe how the annotation tool codes classes in its own `true_col` output, so it can be relabeled onto the map's own coding before comparison. `annotated_path` is where this notebook looks for the single annotated file once STAC Notator work is done.

In [ ]:
# Strata of interest
strata = [0, 1]
STRATUM_LABELS = {0: 'Non-cropland', 1: 'Cropland'}
target_stratum = 1  # cropland: the stratum the final area estimate highlights

# How the annotation tool codes the SAME classes in its own true-label output --
# see Step 2 of area_estimation_bungoma2025.ipynb for the full explanation.
STRATUM_TRUE_LABELS = {'Non-cropland': 'Non-cropland', 'Cropland': 'Cropland'}

# ---- User-defined parameters ----
n_participants = 30              # number of training participants
n_samples_per_participant = 20   # sample units each participant annotates
min_allocation = 5               # floor (sample units) applied to every stratum during proportional allocation
confidence = 0.95                # confidence level used to report the final area estimate's CI
seed = 2025                      # random seed, reused for the sample draw and row shuffle

# Column names expected in the annotation-tool export -- see Step 2 of
# area_estimation_bungoma2025.ipynb for details.
id_col = 'id'
true_col = 'stacnotator_label_name'

# Where to find the single annotated file once STAC Notator work is done --
# update this path if it's saved somewhere other than outputs/.
annotated_path = os.path.join(outputs_path, 'large_groups_sample_annotated.csv')

#### **Step 3.** Pixel counts

>Same as the standard workflow: pixel counting gives each stratum's mapped area (`Area_ha`) and area weight (`Wi`), computed on the equal-area raster from Step 1 so the hectare conversion is unbiased across the whole map.

In [ ]:
pixel_counts_csv = os.path.join(outputs_path, 'pixelcounts_bungoma2025.csv')
pixel_counts = srs.compute_pixel_counts(crop_map_2025_ea, strata=strata, output_csv=pixel_counts_csv)
pixel_counts

#### **Step 4.** Sample size (`Ni`) and proportional allocation

>`Ni`, the total sample size, is simply `n_participants * n_samples_per_participant` -- set directly by classroom capacity rather than derived from a target CV (see the Background section above).
>
>`Ni` is then allocated across strata with the same `allocate_proportional` function the standard notebook uses for its pilot sample: each stratum gets `round(Ni * Wh)` units, at least `min_allocation`.

In [ ]:
Ni = n_participants * n_samples_per_participant
print(f'Total sample size Ni = {n_participants} participants x {n_samples_per_participant} samples/participant = {Ni}')

allocation_csv = os.path.join(outputs_path, 'large_groups_sample_allocation.csv')
allocation = srs.allocate_proportional(
    pixel_counts, n_total=Ni, min_allocation=min_allocation, output_csv=allocation_csv
)
print('Allocation:', allocation)

#### **Step 5.** Draw and export the sample for annotation

>`draw_samples_nested` draws a stratified random sample of pixel-center points, with an independent random stream per stratum (see its docstring in `utils_stratified_random_sampling.py`) -- that property is what lets the standard notebook's pilot and full samples nest, and is harmless but unused here since this notebook only draws once.
>
>As in the standard workflow, rows are shuffled before the exported `id` is assigned, so neither the CSV row order nor the id itself betrays each unit's per-stratum draw order to an interpreter.

In [ ]:
full_gdf = srs.draw_samples_nested(crop_map_2025_ea, allocation, seed=seed, id_prefix='BGM25LG', v=True)

# Shuffle row order first, then assign the exported `id` from that shuffled order.
full_gdf = srs.shuffle_samples(full_gdf, seed=seed)
full_gdf = srs.assign_ids(full_gdf, id_col=id_col, v=True)

# Export the sample (reprojected to EPSG:4326) for photo-interpretation in STAC Notator
sample_csv, _ = srs.export_sample_units(
    full_gdf,
    os.path.join(outputs_path, 'large_groups_sample_for_annotation.csv'),
    stratum_labels=STRATUM_LABELS, id_col=id_col, v=True,
)

> **Checkpoint -- annotation required.**
> Download `outputs/large_groups_sample_for_annotation.csv` and split its rows across the `n_participants` participants (`n_samples_per_participant` rows each). Have each participant interpret their rows in **STAC Notator** against the best available imagery for the 2025 season, recording each unit's *true* class. Combine everyone's results into a single CSV with at least an `id` column (matching `id_col`) and a `true_col` column -- with `true_col` set to `'stacnotator_label_name'`, STAC Notator's own label-name export column can be used directly with no renaming. Save the combined file to `annotated_path` (set in Step 2), then run the next cell.

#### **Step 6.** Load the annotated sample

>The annotation tool's raw `true_col` values are relabeled onto the map's own stratum coding (via `STRATUM_TRUE_LABELS`/`STRATUM_LABELS`, same as the standard notebook) before being read in for area/accuracy estimation. There's no round-2/pilot combination step to do this for us here -- it happens directly on the single annotated file.

In [ ]:
if not os.path.exists(annotated_path):
    raise FileNotFoundError(
        f"Annotated sample not found: {annotated_path}\n"
        "Annotate outputs/large_groups_sample_for_annotation.csv in STAC Notator first, "
        f"save the result with columns ['{id_col}', '{true_col}'] to this path, then re-run this cell."
    )

annotated_raw = pd.read_csv(annotated_path)
annotated_df = srs.relabel_true_stratum(annotated_raw, true_col, STRATUM_LABELS, STRATUM_TRUE_LABELS)

annotated_resolved_path = os.path.join(outputs_path, 'large_groups_sample_annotated_resolved.csv')
annotated_df.to_csv(annotated_resolved_path, index=False)

pred, true, _ = srs.load_full_annotations(annotated_resolved_path, id_col=id_col, stratum_col='stratum', true_col=true_col)
annotated_df.head()

#### **Step 7.** Design-based area estimate

>`compute_stratified_random_sampling_metrics` cross-tabulates each unit's map stratum (`pred`) against its annotated stratum (`true`) and turns the sample counts into an unbiased area estimate per class, with standard error and 95% CI, in hectares and as a percentage of the estimated area (Olofsson et al., 2014, Eqs. 8-10).

In [ ]:
metrics_csv = os.path.join(outputs_path, 'area_estimates_bungoma2025_large_groups.csv')
metrics = srs.compute_stratified_random_sampling_metrics(pixel_counts, pred, true, output_csv=metrics_csv, v=True)
metrics

#### **Step 8.** Map accuracy

>`compute_accuracy_metrics` computes overall accuracy plus each stratum's user's accuracy (Ui -- of the pixels the map calls this class, what fraction really are) and producer's accuracy (Pi -- of the pixels that really are this class, what fraction the map called it), each with SE and 95% CI (Olofsson et al., 2014, Eqs. 1-3, 5-7).

In [ ]:
accuracy_csv = os.path.join(outputs_path, 'accuracy_metrics_bungoma2025_large_groups.csv')
accuracy_metrics, overall_accuracy = srs.compute_accuracy_metrics(pixel_counts, pred, true, output_csv=accuracy_csv, v=True)
accuracy_metrics

### **Result**

In [ ]:
cropland = metrics.loc[target_stratum]

print('=== 2025-season cropland area estimate, Bungoma County (large-group sample design) ===')
print(f"Cropland area: {cropland['Area_ha']:.0f} ha +/- {cropland['CI_Ha']:.0f} ha "
      f"({cropland['CI%'] * 100:.1f}% relative precision at {int(confidence * 100)}% confidence; "
      f"from Ni={Ni} units, not a pre-set CV target)")
print(f"Overall map accuracy: {overall_accuracy['O']:.3f} +/- {overall_accuracy['CI']:.3f}")

This is the final 2025-season cropland area estimate for Bungoma County from the large-group sample design -- one annotation round, sized to the training group rather than to a pre-set accuracy target. For the statistically-optimal two-round pilot + Neyman design, see `area_estimation_bungoma2025.ipynb`.